# Entrainer le modele hybride sur Colab, sans rien d'autre

Piste abandonnee : tout a fini par tourner sur Kaggle. Garde pour memoire.

Ni GitHub, ni Drive : on depose un seul zip (le code, les photos, les labels), le notebook
entraine les trois phases sur le GPU, et rapatrie le modele final a la fin.

Le zip se construit sur le PC :

```powershell
cd dev_ocr/vlm_training
python scripts/zip_selfcontained_colab.py
```

⚠️ Les checkpoints vivent sur le disque de Colab, pas sur Drive. Si la session tombe en cours
de route, ils sont perdus et il faut refaire la phase. C'est precisement le defaut qui a fait
abandonner cette approche au profit de Kaggle.

## Config

In [ ]:
# Which phases to run (set False to skip).
RUN_PHASE_1 = True   # projector warmup (CORD + synthetic) — no photos needed
RUN_PHASE_2 = True   # + LoRA + real photos
RUN_PHASE_3 = True   # low-LR alignment
RUN_EXPORT  = True   # merge LoRA -> single inference checkpoint

FORCE_SMALL_BATCH = False  # set True if you hit CUDA OOM (batch_size 8 -> 4)


## Le GPU est bien la ?

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))


## Deposer le bundle
Run the cell, click **Choose Files**, pick `receipt_vlm_colab_bundle.zip`. ~60 MB may take a minute.

In [ ]:
import io, glob, os, zipfile
from pathlib import Path
from google.colab import files

print("Select receipt_vlm_colab_bundle.zip ...")
uploaded = files.upload()
assert uploaded, "Nothing uploaded"
name = next(iter(uploaded))
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as zf:
    zf.extractall("/content")
print("Unzipped", name)

hits = glob.glob("/content/**/vlm_training/scripts/train.py", recursive=True)
assert hits, "train.py not found — wrong zip?"
TRAIN_PKG = Path(hits[0]).resolve().parents[1]   # .../dev_ocr/vlm_training
DEV_OCR = TRAIN_PKG.parent                        # .../dev_ocr
os.chdir(TRAIN_PKG)
print("Train package:", TRAIN_PKG)


## Installer les dependances

In [ ]:
import subprocess, sys
def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

pip("-r", "requirements-training.txt", "tokenizers>=0.22,<=0.23")
pip("-e", str(DEV_OCR))        # receipt_ocr
pip("-e", str(TRAIN_PKG))      # receipt_vlm
print("Install OK")


## Rediriger les configs vers le disque local
Rewrites `colab_paths.yaml` so checkpoints go to `/content/checkpoints` and real data is read
from the unzipped bundle.

In [ ]:
import yaml
from pathlib import Path

CKPT_DIR = Path("/content/checkpoints"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
cfg_path = TRAIN_PKG / "configs" / "colab_paths.yaml"
cfg = yaml.safe_load(cfg_path.read_text()) or {}
cfg["checkpoint_dir"] = str(CKPT_DIR)
cfg["log_every"] = 25   # batch-loss line every 25 steps (heartbeat)
if FORCE_SMALL_BATCH:
    cfg["batch_size"] = 4
cfg.setdefault("data", {})
cfg["data"]["real_images_dir"] = str(DEV_OCR / "data" / "raw" / "images_tickets_caisse")
cfg["data"]["real_labels_dir"] = str(TRAIN_PKG / "data" / "real_labels")
cfg_path.write_text(yaml.dump(cfg, default_flow_style=False, sort_keys=False))
print(cfg_path.read_text())


## Restaurer les checkpoints apres une deconnexion

Run this **only** if your Colab session restarted and you have `phase*_best.pt`
files saved on your PC (from the backup cell). It uploads them back to
`/content/checkpoints` so you can skip finished phases (set `RUN_PHASE_1/2 = False`
in cell 0). Skip this cell on a normal first run.

In [ ]:
from google.colab import files
from pathlib import Path
Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
print("Choose phase*_best.pt to restore (Cancel to skip) ...")
for name, data in files.upload().items():
    dest = Path(CKPT_DIR) / name
    dest.write_bytes(data)
    print("restored", dest, f"({len(data)/1e9:.2f} GB)")


## Entrainer les trois phases
Phase 1 first downloads CORD (~once) then trains. Watch the per-epoch lines. ~3–4 h total on T4.

In [ ]:
import subprocess, sys, os, time, datetime

p1 = f"{CKPT_DIR}/phase1_best.pt"
p2 = f"{CKPT_DIR}/phase2_best.pt"
p3 = f"{CKPT_DIR}/phase3_best.pt"

def run_train(config, resume=None):
    # -u + PYTHONUNBUFFERED: stream child output live instead of buffering it.
    cmd = [sys.executable, "-u", "scripts/train.py", "--config", config]
    if resume:
        cmd += ["--resume", resume]
    print("
>>", " ".join(cmd), flush=True)
    print("   (startup is silent for a few min: downloading CLIP+SmolLM2, then CORD)", flush=True)

    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    start = last = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    for line in proc.stdout:
        now = time.time()
        gap = now - last; last = now            # seconds since previous line = heartbeat
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"[{ts} +{int(now-start):>5}s gap{gap:4.0f}s] {line}", end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"{config} failed (exit {proc.returncode})")
    print(f"-- {config} done in {int(time.time()-start)}s", flush=True)

# Skipping a phase requires its checkpoint already on disk (use Restore cell 4b).
if not RUN_PHASE_1 and (RUN_PHASE_2 or RUN_PHASE_3):
    assert os.path.exists(p1), f"{p1} missing -- run Restore cell 4b or set RUN_PHASE_1=True"
if not RUN_PHASE_2 and RUN_PHASE_3:
    assert os.path.exists(p2), f"{p2} missing -- run Restore cell 4b or set RUN_PHASE_2=True"

if RUN_PHASE_1:
    run_train("configs/phase1_colab.yaml")
if RUN_PHASE_2:
    run_train("configs/phase2_colab.yaml", resume=p1)
if RUN_PHASE_3:
    run_train("configs/phase3_colab.yaml", resume=p2)
print("Training done")


### Sauvegarder un checkpoint en cours de route
Run this after a phase finishes to download it, so a later disconnect doesn't waste that phase.

In [ ]:
# OPTIONAL insurance: no Drive means /content is wiped on disconnect.
# Checkpoints are adapter-only (~45 MB) so this is fast and reliable.
from google.colab import files
import os
for name in ("phase1_best.pt", "phase2_best.pt", "phase3_best.pt"):
    p = f"{CKPT_DIR}/{name}"
    if os.path.exists(p):
        print("downloading", p, f"({os.path.getsize(p)/1e9:.2f} GB) ...")
        files.download(p)


## Exporter le checkpoint fusionne

In [ ]:
MERGED = f"{CKPT_DIR}/receipt_vlm_500m_merged.pt"
if RUN_EXPORT:
    subprocess.check_call([sys.executable, "scripts/export_checkpoint.py",
                           "--checkpoint", p3, "--output", MERGED])
    print("Merged ->", MERGED)
else:
    print("Export skipped")


## Rapatrier le modele

In [ ]:
from google.colab import files
files.download(MERGED)


## Verification rapide sur une photo

Passe le modele entraine dans le pipeline complet, pour voir ce qu'il sort.

In [ ]:
import json, os
from pathlib import Path

photos = sorted((DEV_OCR / "data" / "raw" / "images_tickets_caisse").glob("*.jpg"))
if photos and Path(MERGED).is_file():
    os.environ.update({
        "RECEIPT_OCR_BACKEND": "vlm",
        "RECEIPT_VLM_MODEL": "receipt-vlm-500m",
        "RECEIPT_VLM_MODE": "json",
        "RECEIPT_VLM_MODEL_PATH": MERGED,
    })
    from receipt_ocr import extract_receipt
    print(json.dumps(extract_receipt(str(photos[0])), indent=2, ensure_ascii=False)[:1200])
else:
    print("Need merged checkpoint + at least one photo")
